In [1]:
!pip install -q pandas numpy scikit-learn statsmodels || pip install -q pandas numpy scikit-learn statsmodels --break-system-packages

# 06. NASA/PROMISE Statistical Significance Testing (McNemar's Test, Bootstrap CIs)

Per the course's Step 4 guidance on statistical significance testing for classifier
comparison: compares the Logistic Regression and Random Forest baseline models trained
in notebook 2 using McNemar's test and bootstrap 95% confidence intervals.


In [2]:
import os as _os_setup
for _d in ["../data/raw", "../data/cleaned", "../figures"]:
    _os_setup.makedirs(_d, exist_ok=True)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from statsmodels.stats.contingency_tables import mcnemar

FEATURES = ['BRANCH_COUNT', 'CYCLOMATIC_COMPLEXITY', 'DESIGN_COMPLEXITY', 'ESSENTIAL_COMPLEXITY',
            'HALSTEAD_CONTENT', 'HALSTEAD_DIFFICULTY', 'HALSTEAD_EFFORT', 'HALSTEAD_ERROR_EST',
            'HALSTEAD_LENGTH', 'HALSTEAD_LEVEL', 'HALSTEAD_PROG_TIME', 'HALSTEAD_VOLUME',
            'LOC_BLANK', 'LOC_CODE_AND_COMMENT', 'LOC_COMMENTS', 'LOC_EXECUTABLE', 'LOC_TOTAL',
            'NUM_OPERANDS', 'NUM_OPERATORS', 'NUM_UNIQUE_OPERANDS', 'NUM_UNIQUE_OPERATORS']

import os
from sklearn.impute import SimpleImputer

CLEAN_CSV = "../data/cleaned/nasa_promise_combined_clean.csv"

if not os.path.exists(CLEAN_CSV):
    import urllib.request
    GITHUB_BASE = "https://raw.githubusercontent.com/daljeetkaurJohar/qm640-governance-analytics/master"
    try:
        print("Not found locally -- trying GitHub repo first (faster than rebuilding)...")
        req = urllib.request.Request(f"{GITHUB_BASE}/data/cleaned/nasa_promise_combined_clean.csv",
                                      headers={"User-Agent": "qm640-capstone"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            content = resp.read()
        os.makedirs("../data/cleaned", exist_ok=True)
        with open(CLEAN_CSV, "wb") as f:
            f.write(content)
        print(f"Downloaded {len(content)} bytes from GitHub -> {CLEAN_CSV}")
    except Exception as e:
        print(f"GitHub fetch failed ({e}) -- will rebuild from source ARFF files instead.")

if not os.path.exists(CLEAN_CSV):
    print("Cleaned CSV not found locally -- regenerating from source ARFF files...")
    import arff, urllib.request

    ARFF_DIR = "../data/raw/nasa_promise_arff"
    ARFF_FILES = ['CM1.arff','KC1.arff','JM1.arff','PC1.arff','PC3.arff','PC4.arff','KC3.arff','MW1.arff']
    GITHUB_RAW_BASE = "https://raw.githubusercontent.com/klainfo/NASADefectDataset/master/CleanedData/MDP/D%27%27"

    os.makedirs(ARFF_DIR, exist_ok=True)
    os.makedirs("../data/cleaned", exist_ok=True)
    for fname in ARFF_FILES:
        fpath = os.path.join(ARFF_DIR, fname)
        if not os.path.exists(fpath):
            print(f"  Downloading real source file: {fname} ...")
            req = urllib.request.Request(f"{GITHUB_RAW_BASE}/{fname}", headers={"User-Agent": "qm640-capstone"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                with open(fpath, "wb") as out:
                    out.write(resp.read())

    def _load_one(fname):
        with open(os.path.join(ARFF_DIR, fname)) as fh:
            d = arff.load(fh)
        cols = [a[0] for a in d['attributes']]
        _df = pd.DataFrame(d['data'], columns=cols)
        label_col = 'Defective' if 'Defective' in _df.columns else 'label'
        _df = _df.rename(columns={label_col: 'Defective'})
        _df['project'] = fname.replace('.arff', '')
        return _df

    _frames = [_load_one(f) for f in ARFF_FILES]
    _raw = pd.concat(_frames, ignore_index=True)
    _raw['Defective_bin'] = (_raw['Defective'] == 'Y').astype(int)

    _clean = _raw.drop_duplicates(subset=FEATURES + ['project']).copy()
    _imputer = SimpleImputer(strategy='median')
    _clean[FEATURES] = _imputer.fit_transform(_clean[FEATURES])
    _clean.to_csv(CLEAN_CSV, index=False)
    print(f"Regenerated {CLEAN_CSV}, shape: {_clean.shape}")
else:
    print(f"Found existing {CLEAN_CSV}, using it directly.")

clean = pd.read_csv("../data/cleaned/nasa_promise_combined_clean.csv")
X = clean[FEATURES].values
y = clean['Defective_bin'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
logreg.fit(X_train_s, y_train)
pred_lr = logreg.predict(X_test_s)
proba_lr = logreg.predict_proba(X_test_s)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
proba_rf = rf.predict_proba(X_test)[:, 1]
print("Re-fit both models on the same train/test split as notebook 2 (random_state=42, identical result)")

Not found locally -- trying GitHub repo first (faster than rebuilding)...
Downloaded 1445029 bytes from GitHub -> ../data/cleaned/nasa_promise_combined_clean.csv
Found existing ../data/cleaned/nasa_promise_combined_clean.csv, using it directly.
Re-fit both models on the same train/test split as notebook 2 (random_state=42, identical result)


## McNemar's Test

In [3]:
lr_correct = (pred_lr == y_test)
rf_correct = (pred_rf == y_test)

n_both_correct = np.sum(lr_correct & rf_correct)
n_lr_only = np.sum(lr_correct & ~rf_correct)
n_rf_only = np.sum(~lr_correct & rf_correct)
n_both_wrong = np.sum(~lr_correct & ~rf_correct)

table = [[n_both_correct, n_lr_only], [n_rf_only, n_both_wrong]]
print(f"Both correct: {n_both_correct}, LR-only-correct: {n_lr_only}, RF-only-correct: {n_rf_only}, Both wrong: {n_both_wrong}")

result = mcnemar(table, exact=False, correction=True)
print(f"McNemar chi-square statistic = {result.statistic:.4f}, p-value = {result.pvalue:.4f}")
if result.pvalue < 0.05:
    print("=> Statistically significant difference between LogReg and RF classification patterns (p < .05)")

Both correct: 1713, LR-only-correct: 100, RF-only-correct: 188, Both wrong: 527
McNemar chi-square statistic = 26.2812, p-value = 0.0000
=> Statistically significant difference between LogReg and RF classification patterns (p < .05)


## Bootstrap 95% Confidence Intervals (2,000 resamples)

In [4]:
rng = np.random.RandomState(42)
n_boot = 2000
n_test = len(y_test)

boot_acc_lr, boot_auc_lr, boot_acc_rf, boot_auc_rf = [], [], [], []

for i in range(n_boot):
    idx = rng.randint(0, n_test, n_test)
    yt = y_test[idx]
    if len(np.unique(yt)) < 2:
        continue
    boot_acc_lr.append(accuracy_score(yt, pred_lr[idx]))
    boot_auc_lr.append(roc_auc_score(yt, proba_lr[idx]))
    boot_acc_rf.append(accuracy_score(yt, pred_rf[idx]))
    boot_auc_rf.append(roc_auc_score(yt, proba_rf[idx]))

def ci(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

lr_acc_ci, lr_auc_ci = ci(boot_acc_lr), ci(boot_auc_lr)
rf_acc_ci, rf_auc_ci = ci(boot_acc_rf), ci(boot_auc_rf)

print(f"Logistic Regression - Accuracy: {np.mean(boot_acc_lr):.3f} [95% CI: {lr_acc_ci[0]:.3f}, {lr_acc_ci[1]:.3f}]")
print(f"Logistic Regression - AUC:      {np.mean(boot_auc_lr):.3f} [95% CI: {lr_auc_ci[0]:.3f}, {lr_auc_ci[1]:.3f}]")
print(f"Random Forest       - Accuracy: {np.mean(boot_acc_rf):.3f} [95% CI: {rf_acc_ci[0]:.3f}, {rf_acc_ci[1]:.3f}]")
print(f"Random Forest       - AUC:      {np.mean(boot_auc_rf):.3f} [95% CI: {rf_auc_ci[0]:.3f}, {rf_auc_ci[1]:.3f}]")

boot_auc_diff = np.array(boot_auc_rf) - np.array(boot_auc_lr)
diff_ci = ci(boot_auc_diff)
print(f"\nPaired bootstrap difference in AUC (RF - LogReg): {np.mean(boot_auc_diff):.4f} [95% CI: {diff_ci[0]:.4f}, {diff_ci[1]:.4f}]")
if diff_ci[0] > 0 or diff_ci[1] < 0:
    print("=> 95% CI excludes zero: RF's AUC advantage is statistically meaningful")
else:
    print("=> 95% CI includes zero: RF's AUC advantage is NOT statistically distinguishable from chance")

Logistic Regression - Accuracy: 0.717 [95% CI: 0.699, 0.734]
Logistic Regression - AUC:      0.719 [95% CI: 0.692, 0.744]
Random Forest       - Accuracy: 0.752 [95% CI: 0.735, 0.769]
Random Forest       - AUC:      0.731 [95% CI: 0.706, 0.755]

Paired bootstrap difference in AUC (RF - LogReg): 0.0126 [95% CI: -0.0024, 0.0267]
=> 95% CI includes zero: RF's AUC advantage is NOT statistically distinguishable from chance


## Interpretation

McNemar's test shows the two models make significantly different per-module
classification decisions (p < .0001), but the bootstrap CI on the AUC difference
includes zero: their overall ability to rank modules by defect risk is not
statistically distinguishable at the 95% confidence level. The choice between the two
models should be driven by the precision/recall trade-off (Random Forest's higher
precision vs. Logistic Regression's higher recall), not by a claim that one model is
unambiguously superior.

In [5]:
import json
stats_summary = {
    "mcnemar_table": [[int(x) for x in row] for row in table],
    "mcnemar_chi2": float(result.statistic), "mcnemar_pvalue": float(result.pvalue),
    "lr_acc_mean": float(np.mean(boot_acc_lr)), "lr_acc_ci": [float(x) for x in lr_acc_ci],
    "lr_auc_mean": float(np.mean(boot_auc_lr)), "lr_auc_ci": [float(x) for x in lr_auc_ci],
    "rf_acc_mean": float(np.mean(boot_acc_rf)), "rf_acc_ci": [float(x) for x in rf_acc_ci],
    "rf_auc_mean": float(np.mean(boot_auc_rf)), "rf_auc_ci": [float(x) for x in rf_auc_ci],
    "auc_diff_mean": float(np.mean(boot_auc_diff)), "auc_diff_ci": [float(x) for x in diff_ci],
}
with open("../data/cleaned/stats_summary.json", "w") as f:
    json.dump(stats_summary, f, indent=2)
print("Saved stats_summary.json")
print(stats_summary)

Saved stats_summary.json
{'mcnemar_table': [[1713, 100], [188, 527]], 'mcnemar_chi2': 26.28125, 'mcnemar_pvalue': 2.9514019251156993e-07, 'lr_acc_mean': 0.7168508702531645, 'lr_acc_ci': [0.6989715189873418, 0.7341772151898734], 'lr_auc_mean': 0.7187211177138615, 'lr_auc_ci': [0.6920205261474838, 0.7437730477972394], 'rf_acc_mean': 0.7520019778481012, 'rf_acc_ci': [0.7345727848101266, 0.7693829113924051], 'rf_auc_mean': 0.7313313071189291, 'rf_auc_ci': [0.7059963107385409, 0.755006038931597], 'auc_diff_mean': 0.012610189405067519, 'auc_diff_ci': [-0.002449913161022588, 0.02665709239661429]}
